In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import math

In [8]:
MODEL_PATH = "models/bmw-gpt2-reduced/original_model"

In [9]:
def calculate_perplexity(model, tokenizer, text):
    encodings = tokenizer(text, return_tensors="pt")
    max_length = model.config.n_positions #The Maximum processable sequence length for GPT-2 is typically 1024
    stride = 512 #Step size (how far the sliding window moves to the right each time)
    seq_len = encodings.input_ids.size(1) #Total length of text

    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        # Calculate the end position of the current window (which cannot exceed the total length of the article).
        end_loc = min(begin_loc + max_length, seq_len)

        # Calculate the length of the “new content” to be predicted.
        trg_len = end_loc - prev_end_loc 

        # Extract the tokens for this segment
        input_ids = encodings.input_ids[:, begin_loc:end_loc]
        
        target_ids = input_ids.clone()
        # Mark “old content viewed in the previous window” as -100
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            # Negative Log-Likelihood Error
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
    # Take the average of the errors across all segments and calculate the exponential.
    ppl = math.exp(torch.stack(nlls).mean())
    return ppl

def generate_text(model, tokenizer, prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt")
    
    output_sequences = model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id, # Tell the model that the zero-padding symbol is the end symbol.
        num_return_sequences=1 # How many results should be generated at once? Just one is sufficient here.
    )
    
    return tokenizer.decode(output_sequences[0], skip_special_tokens=True)

def main():
    print(f"Loading model from {MODEL_PATH}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
    except OSError:
        print("Model not found. Please run train.py first.")
        return

    # 1. Initial questions for qualitative evaluation (subjective test)
    prompts = [
        "The new BMW i7 offers",
        "BMW focuses on sustainability by",
        "The driving performance of the M3 is"
    ]
    
    print("\n--- Qualitative Results (Text Generation) ---")
    # Asking questions in turn
    model.eval()
    for p in prompts:
        generated = generate_text(model, tokenizer, p)
        print(f"\nPROMPT: {p}")
        print(f"RESULT: {generated}")

    # 2. quantitative Evaluation (Perplexity on a sample string) (objective test)
    # Ideally, this is done on the whole test set, but we do a quick check here.
    test_text = "The BMW Group is the world's leading premium manufacturer of automobiles and motorcycles."
    ppl = calculate_perplexity(model, tokenizer, test_text)
    print(f"\n--- Quantitative Results ---")
    print(f"Perplexity on test sentence: {ppl:.2f}")

if __name__ == "__main__":
    main()

Loading model from models/bmw-gpt2-reduced/original_model...

--- Qualitative Results (Text Generation) ---

PROMPT: The new BMW i7 offers
RESULT: The new BMW i7 offers a more compact, fuel efficient and more efficient approach to the road. It combines the power of the i8 with the powerful performance of the i7, providing more power to the car than ever before.

The BMW i7 is available in four petrol, diesel, hybrid and electric modes.

The petrol mode combines the power of the i8 with the performance of the i7, providing more power to the car than ever before.

The hybrid

PROMPT: BMW focuses on sustainability by
RESULT: BMW focuses on sustainability by building a fleet of high-performance electric cars capable of competing with conventional vehicles.
   The company has already launched its first electric car in the USA and has already announced its first fully-electric fleet in the UK.   The UK has a market of over 1.2 billion people, with over half of that market concentrated in the